In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_rg3
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
rg3 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3_다시되돌림.csv")

| 항목 | 확인 결과 |
|---|---:|
| 전체 행 | **1,182** |
| 전체 컬럼 | **29** |
| 정상 | **1,157 (97.88%)** |
| 불량 | **25 (2.12%)** |
| 결측치 | **0개** |
| `original_index` 중복 | **0개** |
| LH | **591** |
| RH | **591** |
| LH 불량 | **0** |
| RH 불량 | **25** |
| 고유 TimeStamp | **591개** |
| TimeStamp당 행 | **정확히 2개 (LH/RH)** |

# 1. 전처리

### 1) 기본 전처리

In [4]:
# 시간형 변환
rg3["TimeStamp"] = pd.to_datetime(rg3["TimeStamp"])

# 시간 순 정렬
rg3 = rg3.sort_values(
    ["TimeStamp", "original_index"]
).reset_index(drop=True)

# RH = 1, LH = 0
rg3["Side_RH"] = (rg3["RH_LH"] == "RH").astype(int)

# 제거할 컬럼
rg3 = rg3.drop(
    columns=[
        "Clamp_Open_Position",  # 완전 상수
        "PART_NAME"             # RH_LH와 완전 중복
    ]
)

rg3.shape

(1182, 28)

### 2) 상수/저고유값 변수 점검

In [5]:
# 수치형 변수별 고유값 개수 확인

exclude_cols = [
    "original_index",
    "PassOrFail",
    "Side_RH"
]

numeric_cols = rg3.select_dtypes(include="number").columns
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

unique_summary = (
    rg3[feature_cols]
    .nunique()
    .sort_values()
    .to_frame("nunique")
)

unique_summary

,nunique
Filling_Time,2
Injection_Time,3
Clamp_Close_Time,3
Max_Screw_RPM,4
Average_Screw_RPM,4
Cushion_Position,7
Max_Injection_Speed,7
Plasticizing_Position,8
Max_Injection_Pressure,10
Cycle_Time,10


### 3) 동일 TimeStamp의 LH/RH 구조와 Target 관계

In [6]:
# LH / RH별 정상·불량 분포 확인

pd.crosstab(
    rg3["RH_LH"],
    rg3["PassOrFail"],
    margins=True
)

PassOrFail,0,1,All
RH_LH,,,
LH,591,0,591
RH,566,25,591
All,1157,25,1182


RG3는 일반적인 행 단위 데이터로 그대로 모델링하면 안됨.   

25개 cycle 전부   
동일 공정조건 X   
 ├─ LH → 정상(0)   
 └─ RH → 불량(1)   

In [7]:
# 같은 TimeStamp 안에서 Target이 서로 다른 경우 확인

target_conflict = (
    rg3.groupby("TimeStamp")["PassOrFail"]
    .nunique()
)

target_conflict.value_counts()

PassOrFail
1    566
2     25
Name: count, dtype: int64

# 4. 첫번째 시도) RH만 남겨서 cycle당 1행으로 모델링

- 센서값은 LH/RH가 완전히 동일   
- LH 591개에는 불량이 단 한 건도 없음   
- RH에는 정상 566 / 불량 25   
- LH를 추가한다고 공정정보가 늘어나지 않음   
- 오히려 정상 데이터만 591개 중복 추가되어 클래스 불균형 악화   
- Side_RH를 넣으면 모델이 "LH면 무조건 정상"이라는 shortcut을 배우기 쉬움   

### 1) RH 기준 데이터셋 생성

In [8]:
rg3_rh = (
    rg3[rg3["RH_LH"] == "RH"]
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)

rg3_rh.shape

(591, 28)

In [9]:
rg3_rh["PassOrFail"].value_counts()

PassOrFail
0    566
1     25
Name: count, dtype: int64

In [10]:
rg3_rh["PassOrFail"].value_counts(normalize=True) * 100

PassOrFail
0    95.769882
1     4.230118
Name: proportion, dtype: float64

In [11]:
drop_cols = [
    "PART_NAME",
    "RH_LH",
    "Side_RH"
]

rg3_rh = rg3_rh.drop(
    columns=[col for col in drop_cols if col in rg3_rh.columns]
)

rg3_rh.shape

(591, 26)

### 2) 생산 간격 확인

In [12]:
rg3_rh["gap_min"] = (
    rg3_rh["TimeStamp"]
    .diff()
    .dt.total_seconds() / 60
)

rg3_rh["gap_min"].describe()

count     590.000000
mean        5.643418
std        71.607218
min         1.000000
25%         1.016667
50%         1.033333
75%         1.050000
max      1406.466667
Name: gap_min, dtype: float64

In [13]:
rg3_rh[rg3_rh["gap_min"] > 5][
    ["TimeStamp", "gap_min", "PassOrFail"]
]

,TimeStamp,gap_min,PassOrFail
41,2020-10-22 00:24:05,1406.466667,0
105,2020-10-22 01:40:17,11.350000,0
144,2020-10-22 04:40:26,140.000000,0
341,2020-10-23 01:01:28,1012.066667,0
369,2020-10-23 01:47:49,18.533333,0
415,2020-10-23 04:41:48,125.616667,0


5분 이상 끊겼으면 새로운 생산 세션 정의

In [14]:
# 5분 초과 공백이면 새로운 생산 세션

rg3_rh["session_id"] = (
    (rg3_rh["gap_min"] > 5)
    .cumsum()
)

rg3_rh["session_id"].value_counts().sort_index()

session_id
0     41
1     64
2     39
3    197
4     28
5     46
6    176
Name: count, dtype: int64

In [15]:
session_summary = rg3_rh.groupby("session_id").agg(
    start_time=("TimeStamp", "min"),
    end_time=("TimeStamp", "max"),
    count=("PassOrFail", "size"),
    defects=("PassOrFail", "sum")
)

session_summary

,start_time,end_time,count,defects
session_id,,,,
0,2020-10-21 00:16:26,2020-10-21 00:57:37,41,0
1,2020-10-22 00:24:05,2020-10-22 01:28:56,64,4
2,2020-10-22 01:40:17,2020-10-22 02:20:26,39,1
3,2020-10-22 04:40:26,2020-10-22 08:09:24,197,10
4,2020-10-23 01:01:28,2020-10-23 01:29:17,28,2
5,2020-10-23 01:47:49,2020-10-23 02:36:11,46,2
6,2020-10-23 04:41:48,2020-10-23 07:46:03,176,6


* 파생변수 만들 때 아래처럼 할 거임   

세션 0   
제품1 → 제품2 → 제품3 → ...   
   
       생산중단   
   
세션 1   
제품1 → 제품2 → 제품3 → ...   

### 3) 불량 위치 확인

In [16]:
# 세션 내 생산 순번
rg3_rh["cycle_in_session"] = (
    rg3_rh.groupby("session_id")
    .cumcount()
)

# 불량 행만 확인
defect_positions = rg3_rh.loc[
    rg3_rh["PassOrFail"] == 1,
    ["TimeStamp", "session_id", "cycle_in_session", "gap_min", "PassOrFail"]
].copy()

# 같은 세션 내 직전 불량과 몇 cycle 차이인지
defect_positions["prev_defect_cycle_gap"] = (
    defect_positions.groupby("session_id")["cycle_in_session"]
    .diff()
)

defect_positions

,TimeStamp,session_id,cycle_in_session,gap_min,PassOrFail,prev_defect_cycle_gap
68,2020-10-22 00:51:52,1,27,1.000000,1,NaN
80,2020-10-22 01:04:15,1,39,1.033333,1,12.0
85,2020-10-22 01:09:23,1,44,1.033333,1,5.0
102,2020-10-22 01:26:54,1,61,1.050000,1,17.0
122,2020-10-22 01:57:47,2,17,1.050000,1,NaN
150,2020-10-22 04:46:36,3,6,1.050000,1,NaN
152,2020-10-22 04:48:39,3,8,1.016667,1,2.0
155,2020-10-22 04:51:46,3,11,1.050000,1,3.0
161,2020-10-22 04:57:56,3,17,1.033333,1,6.0
173,2020-10-22 05:10:17,3,29,1.033333,1,12.0


In [17]:
defect_positions.groupby("session_id").agg(
    defect_count=("PassOrFail", "size"),
    first_defect_cycle=("cycle_in_session", "min"),
    last_defect_cycle=("cycle_in_session", "max"),
    avg_defect_gap=("prev_defect_cycle_gap", "mean"),
    min_defect_gap=("prev_defect_cycle_gap", "min"),
    max_defect_gap=("prev_defect_cycle_gap", "max")
)

,defect_count,first_defect_cycle,last_defect_cycle,avg_defect_gap,min_defect_gap,max_defect_gap
session_id,,,,,,
1,4,27,61,11.333333,5.0,17.0
2,1,17,17,NaN,NaN,NaN
3,10,6,166,17.777778,2.0,44.0
4,2,2,22,20.000000,20.0,20.0
5,2,12,36,24.000000,24.0,24.0
6,6,32,151,23.800000,3.0,74.0


In [18]:
# 센서 컬럼 분리

# 모델 입력에서 제외할 메타 컬럼
exclude_cols = [
    "original_index",
    "TimeStamp",
    "PassOrFail",
    "gap_min",
    "session_id",
    "cycle_in_session"
]

sensor_cols = [
    col for col in rg3_rh.select_dtypes(include="number").columns
    if col not in exclude_cols
]

print("센서 컬럼 수:", len(sensor_cols))
print(sensor_cols)

센서 컬럼 수: 23
['Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Plasticizing_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']


### 4) lag 생성

In [19]:
for col in sensor_cols:
    for lag in [1, 2, 3]:
        rg3_rh[f"{col}_lag{lag}"] = (
            rg3_rh.groupby("session_id")[col]
            .shift(lag)
        )

In [20]:
# 직전 cycle 대비 변화량

for col in sensor_cols:
    rg3_rh[f"{col}_diff1"] = (
        rg3_rh.groupby("session_id")[col]
        .diff(1)
    )

In [21]:
# rolling 

for col in sensor_cols:
    for window in [3, 5, 10]:

        rg3_rh[f"{col}_roll{window}_mean"] = (
            rg3_rh.groupby("session_id")[col]
            .transform(
                lambda x: x.rolling(
                    window=window,
                    min_periods=1
                ).mean()
            )
        )

        rg3_rh[f"{col}_roll{window}_std"] = (
            rg3_rh.groupby("session_id")[col]
            .transform(
                lambda x: x.rolling(
                    window=window,
                    min_periods=2
                ).std()
            )
        )

C:\Users\Administrator\AppData\Local\Temp\ipykernel_29516\476207820.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  rg3_rh[f"{col}_roll{window}_mean"] = (
C:\Users\Administrator\AppData\Local\Temp\ipykernel_29516\476207820.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  rg3_rh[f"{col}_roll{window}_std"] = (


In [22]:
print("전체 shape:", rg3_rh.shape)

print("\n결측치 많은 컬럼 TOP 20")
display(
    rg3_rh.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

전체 shape: (591, 259)

결측치 많은 컬럼 TOP 20


Injection_Time_lag3            21
Clamp_Close_Time_lag3          21
Cycle_Time_lag3                21
Plasticizing_Time_lag3         21
Cushion_Position_lag3          21
Barrel_Temperature_5_lag3      21
Average_Back_Pressure_lag3     21
Barrel_Temperature_1_lag3      21
Barrel_Temperature_2_lag3      21
Max_Injection_Speed_lag3       21
Plasticizing_Position_lag3     21
Barrel_Temperature_3_lag3      21
Max_Screw_RPM_lag3             21
Average_Screw_RPM_lag3         21
Max_Injection_Pressure_lag3    21
Mold_Temperature_4_lag3        21
Barrel_Temperature_4_lag3      21
Max_Back_Pressure_lag3         21
Mold_Temperature_3_lag3        21
Barrel_Temperature_6_lag3      21
dtype: int64

### 4) 원본 / Lag / Diff / Rolling 비교

In [23]:
# 파생변수 그룹 구분

original_cols = sensor_cols

lag1_cols = [
    c for c in rg3_rh.columns
    if c.endswith("_lag1")
]

lag_cols = [
    c for c in rg3_rh.columns
    if "_lag" in c
]

diff_cols = [
    c for c in rg3_rh.columns
    if c.endswith("_diff1")
]

roll3_cols = [
    c for c in rg3_rh.columns
    if "_roll3_" in c
]

roll5_cols = [
    c for c in rg3_rh.columns
    if "_roll5_" in c
]

roll10_cols = [
    c for c in rg3_rh.columns
    if "_roll10_" in c
]

print("Original :", len(original_cols))
print("Lag1     :", len(lag1_cols))
print("Lag 전체 :", len(lag_cols))
print("Diff1    :", len(diff_cols))
print("Roll3    :", len(roll3_cols))
print("Roll5    :", len(roll5_cols))
print("Roll10   :", len(roll10_cols))

Original : 23
Lag1     : 23
Lag 전체 : 69
Diff1    : 23
Roll3    : 46
Roll5    : 46
Roll10   : 46


In [24]:
feature_sets = {
    "Original":
        original_cols,

    "Original + Lag1":
        original_cols + lag1_cols,

    "Original + Lag1~3":
        original_cols + lag_cols,

    "Original + Diff1":
        original_cols + diff_cols,

    "Original + Roll3":
        original_cols + roll3_cols,

    "Original + Roll3+5":
        original_cols + roll3_cols + roll5_cols,

    "Original + Lag + Diff":
        original_cols + lag_cols + diff_cols,

    "Original + Lag + Diff + Roll3+5":
        original_cols
        + lag_cols
        + diff_cols
        + roll3_cols
        + roll5_cols,

    "ALL":
        original_cols
        + lag_cols
        + diff_cols
        + roll3_cols
        + roll5_cols
        + roll10_cols
}

In [25]:
feature_sets = {
    name: list(dict.fromkeys(cols))
    for name, cols in feature_sets.items()
}

In [26]:
time_splits = [
    ([0, 1, 2],       [3]),
    ([0, 1, 2, 3],    [4]),
    ([0, 1, 2, 3, 4], [5]),
    ([0, 1, 2, 3, 4, 5], [6]),
]

In [27]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

In [28]:
results = []

for set_name, features in feature_sets.items():

    fold_pr = []
    fold_roc = []

    for fold, (train_sessions, valid_sessions) in enumerate(
        time_splits,
        start=1
    ):

        train_df = rg3_rh[
            rg3_rh["session_id"].isin(train_sessions)
        ]

        valid_df = rg3_rh[
            rg3_rh["session_id"].isin(valid_sessions)
        ]

        X_train = train_df[features]
        y_train = train_df["PassOrFail"]

        X_valid = valid_df[features]
        y_valid = valid_df["PassOrFail"]

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=5,
                    min_samples_leaf=3,
                    max_features="sqrt",
                    class_weight="balanced_subsample",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        model.fit(X_train, y_train)

        pred_proba = model.predict_proba(X_valid)[:, 1]

        pr = average_precision_score(
            y_valid,
            pred_proba
        )

        roc = roc_auc_score(
            y_valid,
            pred_proba
        )

        fold_pr.append(pr)
        fold_roc.append(roc)

    results.append({
        "Feature_Set": set_name,
        "N_Features": len(features),

        "PR_Mean": np.mean(fold_pr),
        "PR_Min": np.min(fold_pr),
        "PR_Max": np.max(fold_pr),

        "ROC_Mean": np.mean(fold_roc),
        "ROC_Min": np.min(fold_roc),
        "ROC_Max": np.max(fold_roc),
    })

In [29]:
feature_compare = (
    pd.DataFrame(results)
    .sort_values(
        ["PR_Mean", "PR_Min"],
        ascending=False
    )
    .reset_index(drop=True)
)

feature_compare

,Feature_Set,N_Features,PR_Mean,PR_Min,PR_Max,ROC_Mean,ROC_Min,ROC_Max
0,ALL,253,0.185018,0.054233,0.521739,0.446947,0.000000,0.769608
1,Original + Roll3,69,0.092823,0.034897,0.228429,0.352325,0.079545,0.789216
2,Original + Roll3+5,115,0.073671,0.052668,0.111580,0.468217,0.326923,0.775490
3,Original + Lag + Diff + Roll3+5,207,0.072964,0.046218,0.130558,0.391313,0.115385,0.661765
4,Original,23,0.069769,0.041617,0.130265,0.357145,0.153846,0.641176
5,Original + Lag + Diff,115,0.059141,0.048519,0.077295,0.348168,0.019231,0.515686
6,Original + Lag1,46,0.056851,0.048745,0.071739,0.363872,0.076923,0.563725
7,Original + Diff1,46,0.056254,0.045571,0.078788,0.402882,0.306818,0.597059
8,Original + Lag1~3,92,0.055392,0.044989,0.067194,0.363984,0.115385,0.539216


In [30]:
fold_results = []

for set_name, features in feature_sets.items():

    for fold, (train_sessions, valid_sessions) in enumerate(
        time_splits,
        start=1
    ):

        train_df = rg3_rh[
            rg3_rh["session_id"].isin(train_sessions)
        ]

        valid_df = rg3_rh[
            rg3_rh["session_id"].isin(valid_sessions)
        ]

        X_train = train_df[features]
        y_train = train_df["PassOrFail"]

        X_valid = valid_df[features]
        y_valid = valid_df["PassOrFail"]

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=5,
                    min_samples_leaf=3,
                    max_features="sqrt",
                    class_weight="balanced_subsample",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])

        model.fit(X_train, y_train)

        pred = model.predict_proba(X_valid)[:, 1]

        fold_results.append({
            "Feature_Set": set_name,
            "Fold": fold,
            "Train_Sessions": str(train_sessions),
            "Valid_Session": valid_sessions[0],
            "Valid_N": len(valid_df),
            "Valid_Defects": int(y_valid.sum()),
            "Baseline_PR": y_valid.mean(),
            "PR_AUC": average_precision_score(
                y_valid,
                pred
            ),
            "ROC_AUC": roc_auc_score(
                y_valid,
                pred
            )
        })

fold_detail = pd.DataFrame(fold_results)

fold_detail.sort_values(
    ["Feature_Set", "Fold"]
)

,Feature_Set,Fold,Train_Sessions,Valid_Session,Valid_N,Valid_Defects,Baseline_PR,PR_AUC,ROC_AUC
32,ALL,1,"[0, 1, 2]",3,197,10,0.050761,0.054907,0.518182
33,ALL,2,"[0, 1, 2, 3]",4,28,2,0.071429,0.054233,0.000000
34,ALL,3,"[0, 1, 2, 3, 4]",5,46,2,0.043478,0.521739,0.500000
35,ALL,4,"[0, 1, 2, 3, 4, 5]",6,176,6,0.034091,0.109193,0.769608
0,Original,1,"[0, 1, 2]",3,197,10,0.050761,0.041617,0.372193
1,Original,2,"[0, 1, 2, 3]",4,28,2,0.071429,0.062727,0.153846
2,Original,3,"[0, 1, 2, 3, 4]",5,46,2,0.043478,0.044466,0.261364
3,Original,4,"[0, 1, 2, 3, 4, 5]",6,176,6,0.034091,0.130265,0.641176
12,Original + Diff1,1,"[0, 1, 2]",3,197,10,0.050761,0.045571,0.361497
13,Original + Diff1,2,"[0, 1, 2, 3]",4,28,2,0.071429,0.078788,0.346154


In [31]:
fold_detail[
    fold_detail["Feature_Set"].isin([
        "Original",
        "Original + Roll3",
        "Original + Roll3+5",
        "ALL"
    ])
][
    [
        "Feature_Set",
        "Fold",
        "Valid_Session",
        "Valid_N",
        "Valid_Defects",
        "Baseline_PR",
        "PR_AUC",
        "ROC_AUC"
    ]
]

,Feature_Set,Fold,Valid_Session,Valid_N,Valid_Defects,Baseline_PR,PR_AUC,ROC_AUC
0,Original,1,3,197,10,0.050761,0.041617,0.372193
1,Original,2,4,28,2,0.071429,0.062727,0.153846
2,Original,3,5,46,2,0.043478,0.044466,0.261364
3,Original,4,6,176,6,0.034091,0.130265,0.641176
16,Original + Roll3,1,3,197,10,0.050761,0.048672,0.444385
17,Original + Roll3,2,4,28,2,0.071429,0.059295,0.096154
18,Original + Roll3,3,5,46,2,0.043478,0.034897,0.079545
19,Original + Roll3,4,6,176,6,0.034091,0.228429,0.789216
20,Original + Roll3+5,1,3,197,10,0.050761,0.053624,0.418182
21,Original + Roll3+5,2,4,28,2,0.071429,0.076812,0.326923


In [32]:
candidate_features = feature_sets["Original + Roll3+5"]

print("후보 Feature 수:", len(candidate_features))

후보 Feature 수: 115


In [33]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score

top_k_list = [10, 20, 30, 40, 60, 80, 115]

topk_results = []
selected_feature_records = []

for k in top_k_list:

    for fold, (train_sessions, valid_sessions) in enumerate(
        time_splits,
        start=1
    ):

        train_df = rg3_rh[
            rg3_rh["session_id"].isin(train_sessions)
        ].copy()

        valid_df = rg3_rh[
            rg3_rh["session_id"].isin(valid_sessions)
        ].copy()

        X_train = train_df[candidate_features]
        y_train = train_df["PassOrFail"]

        X_valid = valid_df[candidate_features]
        y_valid = valid_df["PassOrFail"]

        # train 기준으로만 결측치 처리
        imputer = SimpleImputer(strategy="median")

        X_train_imp = imputer.fit_transform(X_train)
        X_valid_imp = imputer.transform(X_valid)

        # ----------------------------
        # 1. Train 데이터에서 Feature 중요도 계산
        # ----------------------------
        selector = RandomForestClassifier(
            n_estimators=500,
            max_depth=5,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        )

        selector.fit(
            X_train_imp,
            y_train
        )

        importance = pd.Series(
            selector.feature_importances_,
            index=candidate_features
        ).sort_values(ascending=False)

        selected = importance.head(k).index.tolist()

        selected_idx = [
            candidate_features.index(c)
            for c in selected
        ]

        # ----------------------------
        # 2. 선택된 Feature만 사용해 모델 재학습
        # ----------------------------
        model = RandomForestClassifier(
            n_estimators=500,
            max_depth=5,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train_imp[:, selected_idx],
            y_train
        )

        pred = model.predict_proba(
            X_valid_imp[:, selected_idx]
        )[:, 1]

        baseline_pr = y_valid.mean()

        pr = average_precision_score(
            y_valid,
            pred
        )

        roc = roc_auc_score(
            y_valid,
            pred
        )

        topk_results.append({
            "Top_K": k,
            "Fold": fold,
            "Valid_Session": valid_sessions[0],
            "N_Valid": len(y_valid),
            "Defects": int(y_valid.sum()),
            "Baseline_PR": baseline_pr,
            "PR_AUC": pr,
            "PR_Uplift": pr - baseline_pr,
            "ROC_AUC": roc
        })

        # 어떤 Feature가 계속 선택되는지도 저장
        for rank, feature in enumerate(selected, start=1):
            selected_feature_records.append({
                "Top_K": k,
                "Fold": fold,
                "Rank": rank,
                "Feature": feature,
                "Importance": importance[feature]
            })

In [34]:
topk_df = pd.DataFrame(topk_results)

topk_summary = (
    topk_df.groupby("Top_K")
    .agg(
        PR_Mean=("PR_AUC", "mean"),
        PR_Min=("PR_AUC", "min"),
        PR_Max=("PR_AUC", "max"),
        Uplift_Mean=("PR_Uplift", "mean"),
        Uplift_Min=("PR_Uplift", "min"),
        ROC_Mean=("ROC_AUC", "mean")
    )
    .reset_index()
    .sort_values(
        ["Uplift_Mean", "Uplift_Min"],
        ascending=False
    )
)

topk_summary

,Top_K,PR_Mean,PR_Min,PR_Max,Uplift_Mean,Uplift_Min,ROC_Mean
2,30,0.108097,0.053977,0.141830,0.058157,0.010499,0.468736
1,20,0.095762,0.047222,0.141137,0.045822,-0.011905,0.371092
4,60,0.089281,0.060105,0.165915,0.039342,-0.009398,0.423823
3,40,0.088392,0.053977,0.138265,0.038453,0.005495,0.430834
0,10,0.075575,0.047318,0.147022,0.025635,-0.013975,0.364875
5,80,0.074524,0.043961,0.113317,0.024584,0.000483,0.417492
6,115,0.060500,0.046835,0.081985,0.010560,-0.009398,0.393341


In [35]:
selected_df = pd.DataFrame(selected_feature_records)

feature_stability = (
    selected_df[
        selected_df["Top_K"] == 30
    ]
    .groupby("Feature")
    .agg(
        Selected_Count=("Feature", "size"),
        Mean_Rank=("Rank", "mean"),
        Mean_Importance=("Importance", "mean")
    )
    .sort_values(
        ["Selected_Count", "Mean_Rank"],
        ascending=[False, True]
    )
)

feature_stability.head(30)

,Selected_Count,Mean_Rank,Mean_Importance
Feature,,,
Barrel_Temperature_5_roll3_mean,4,4.500000,0.029736
Barrel_Temperature_3_roll3_std,4,6.750000,0.027874
Barrel_Temperature_6_roll5_mean,4,8.000000,0.024160
Barrel_Temperature_3_roll5_mean,4,9.750000,0.022515
Barrel_Temperature_5_roll5_mean,4,13.000000,0.018201
Plasticizing_Time_roll5_mean,4,14.000000,0.019458
Max_Injection_Pressure_roll5_std,4,16.750000,0.016830
Cushion_Position_roll5_std,4,18.250000,0.016930
Hopper_Temperature_roll3_mean,3,1.666667,0.034094


| Feature | 안정성 | 해석 |
|---|---:|---|
| `Barrel_Temperature_5_roll3_mean` | 4/4 | 배럴 5구간 단기 온도 수준 |
| `Barrel_Temperature_3_roll3_std` | 4/4 | 배럴 3구간 단기 온도 흔들림 |
| `Barrel_Temperature_6_roll5_mean` | 4/4 | 배럴 후단 온도 수준 |
| `Barrel_Temperature_3_roll5_mean` | 4/4 | 배럴 3구간 온도 추세 |
| `Barrel_Temperature_5_roll5_mean` | 4/4 | 배럴 5구간 온도 추세 |
| `Plasticizing_Time_roll5_mean` | 4/4 | 최근 가소화 시간 수준 |
| `Max_Injection_Pressure_roll5_std` | 4/4 | 사출 최대압력 변동성 |
| `Cushion_Position_roll5_std` | 4/4 | 쿠션 위치 변동성 |

In [36]:
stable_features = (
    feature_stability[
        feature_stability["Selected_Count"] >= 3
    ]
    .index
    .tolist()
)

print("Stable Feature 수:", len(stable_features))
stable_features

Stable Feature 수: 21


['Barrel_Temperature_5_roll3_mean',
 'Barrel_Temperature_3_roll3_std',
 'Barrel_Temperature_6_roll5_mean',
 'Barrel_Temperature_3_roll5_mean',
 'Barrel_Temperature_5_roll5_mean',
 'Plasticizing_Time_roll5_mean',
 'Max_Injection_Pressure_roll5_std',
 'Cushion_Position_roll5_std',
 'Hopper_Temperature_roll3_mean',
 'Barrel_Temperature_5_roll5_std',
 'Plasticizing_Position_roll5_std',
 'Average_Back_Pressure_roll5_std',
 'Mold_Temperature_3_roll5_std',
 'Barrel_Temperature_2_roll5_std',
 'Barrel_Temperature_3_roll5_std',
 'Hopper_Temperature',
 'Barrel_Temperature_4_roll3_mean',
 'Mold_Temperature_4_roll5_mean',
 'Max_Injection_Pressure_roll3_mean',
 'Max_Switch_Over_Pressure_roll3_mean',
 'Max_Switch_Over_Pressure_roll5_std']

In [37]:
stable_results = []

for fold, (train_sessions, valid_sessions) in enumerate(
    time_splits,
    start=1
):

    train_df = rg3_rh[
        rg3_rh["session_id"].isin(train_sessions)
    ]

    valid_df = rg3_rh[
        rg3_rh["session_id"].isin(valid_sessions)
    ]

    X_train = train_df[stable_features]
    y_train = train_df["PassOrFail"]

    X_valid = valid_df[stable_features]
    y_valid = valid_df["PassOrFail"]

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=5,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    model.fit(X_train, y_train)

    pred = model.predict_proba(X_valid)[:, 1]

    baseline = y_valid.mean()

    pr = average_precision_score(
        y_valid,
        pred
    )

    roc = roc_auc_score(
        y_valid,
        pred
    )

    stable_results.append({
        "Fold": fold,
        "Valid_Session": valid_sessions[0],
        "N": len(y_valid),
        "Defects": int(y_valid.sum()),
        "Baseline_PR": baseline,
        "PR_AUC": pr,
        "PR_Uplift": pr - baseline,
        "ROC_AUC": roc
    })

stable_result_df = pd.DataFrame(stable_results)

display(stable_result_df)

print(
    "\nPR Mean:",
    stable_result_df["PR_AUC"].mean()
)

print(
    "PR Min:",
    stable_result_df["PR_AUC"].min()
)

print(
    "ROC Mean:",
    stable_result_df["ROC_AUC"].mean()
)

,Fold,Valid_Session,N,Defects,Baseline_PR,PR_AUC,PR_Uplift,ROC_AUC
0,1,3,197,10,0.050761,0.081953,0.031192,0.606952
1,2,4,28,2,0.071429,0.120370,0.048942,0.423077
2,3,5,46,2,0.043478,0.106589,0.063111,0.477273
3,4,6,176,6,0.034091,0.064513,0.030422,0.650980



PR Mean: 0.09335634893091822
PR Min: 0.06451252543584667
ROC Mean: 0.5395704785410668


| 지표 | 결과 | 판단 |
|---|---:|---|
| 평균 Baseline PR | 약 **0.050** | 랜덤 기준 |
| Stable PR-AUC | **0.093** | 약 **1.87배** |
| PR 최소 | **0.0645** | 모든 Fold에서 baseline 이상 |
| ROC-AUC 평균 | **0.540** | 전체 분리력은 약함 |

In [38]:
candidate_features = feature_sets["Original + Roll3+5"]

clean_results = []
clean_selected = []

for fold, (train_sessions, valid_sessions) in enumerate(
    time_splits,
    start=1
):

    train_df = rg3_rh[
        rg3_rh["session_id"].isin(train_sessions)
    ].copy()

    valid_df = rg3_rh[
        rg3_rh["session_id"].isin(valid_sessions)
    ].copy()

    X_train = train_df[candidate_features]
    y_train = train_df["PassOrFail"]

    X_valid = valid_df[candidate_features]
    y_valid = valid_df["PassOrFail"]

    # -------------------------
    # 결측치 처리
    # train 기준으로만 fit
    # -------------------------
    imputer = SimpleImputer(strategy="median")

    X_train_imp = imputer.fit_transform(X_train)
    X_valid_imp = imputer.transform(X_valid)

    # -------------------------
    # Feature Selection
    # train 데이터만 사용
    # -------------------------
    selector = RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )

    selector.fit(
        X_train_imp,
        y_train
    )

    importance = pd.Series(
        selector.feature_importances_,
        index=candidate_features
    ).sort_values(ascending=False)

    # K = 30 고정
    selected_features = (
        importance
        .head(30)
        .index
        .tolist()
    )

    selected_idx = [
        candidate_features.index(c)
        for c in selected_features
    ]

    # -------------------------
    # 최종 모델
    # -------------------------
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_imp[:, selected_idx],
        y_train
    )

    pred = model.predict_proba(
        X_valid_imp[:, selected_idx]
    )[:, 1]

    baseline = y_valid.mean()

    pr = average_precision_score(
        y_valid,
        pred
    )

    roc = roc_auc_score(
        y_valid,
        pred
    )

    clean_results.append({
        "Fold": fold,
        "Train_Sessions": str(train_sessions),
        "Valid_Session": valid_sessions[0],
        "N": len(y_valid),
        "Defects": int(y_valid.sum()),
        "Baseline_PR": baseline,
        "PR_AUC": pr,
        "PR_Uplift": pr - baseline,
        "ROC_AUC": roc
    })

    for rank, feature in enumerate(
        selected_features,
        start=1
    ):
        clean_selected.append({
            "Fold": fold,
            "Rank": rank,
            "Feature": feature
        })

In [39]:
clean_result_df = pd.DataFrame(clean_results)

display(clean_result_df)

print(
    "Baseline Mean:",
    clean_result_df["Baseline_PR"].mean()
)

print(
    "PR Mean:",
    clean_result_df["PR_AUC"].mean()
)

print(
    "PR Min:",
    clean_result_df["PR_AUC"].min()
)

print(
    "Uplift Mean:",
    clean_result_df["PR_Uplift"].mean()
)

print(
    "ROC Mean:",
    clean_result_df["ROC_AUC"].mean()
)

,Fold,Train_Sessions,Valid_Session,N,Defects,Baseline_PR,PR_AUC,PR_Uplift,ROC_AUC
0,1,"[0, 1, 2]",3,197,10,0.050761,0.141830,0.091069,0.439037
1,2,"[0, 1, 2, 3]",4,28,2,0.071429,0.107143,0.035714,0.384615
2,3,"[0, 1, 2, 3, 4]",5,46,2,0.043478,0.053977,0.010499,0.352273
3,4,"[0, 1, 2, 3, 4, 5]",6,176,6,0.034091,0.129437,0.095346,0.699020


Baseline Mean: 0.04993979067721067
PR Mean: 0.10809677538421754
PR Min: 0.05397727272727273
Uplift Mean: 0.05815698470700687
ROC Mean: 0.4687362882215823


In [40]:
clean_selected_df = pd.DataFrame(clean_selected)

clean_stability = (
    clean_selected_df
    .groupby("Feature")
    .agg(
        Selected_Count=("Feature", "size"),
        Mean_Rank=("Rank", "mean")
    )
    .sort_values(
        ["Selected_Count", "Mean_Rank"],
        ascending=[False, True]
    )
)

clean_stability.head(30)

,Selected_Count,Mean_Rank
Feature,,
Barrel_Temperature_5_roll3_mean,4,4.500000
Barrel_Temperature_3_roll3_std,4,6.750000
Barrel_Temperature_6_roll5_mean,4,8.000000
Barrel_Temperature_3_roll5_mean,4,9.750000
Barrel_Temperature_5_roll5_mean,4,13.000000
Plasticizing_Time_roll5_mean,4,14.000000
Max_Injection_Pressure_roll5_std,4,16.750000
Cushion_Position_roll5_std,4,18.250000
Hopper_Temperature_roll3_mean,3,1.666667


| 지표 | 결과 | 해석 |
|---|---:|---|
| Baseline PR | **0.0499** | 불량률 기준 |
| PR-AUC | **0.1081** | baseline의 약 **2.16배** |
| PR Uplift | **+0.0582** | 모든 Fold에서 양수 |
| ROC-AUC | **0.4687** | 전체적인 정상/불량 분리 실패 |

# 6. 원본에서 추가할 컬럼 다시 보기

### 1) 원본과 현재 rg3 컬럼 비교

In [2]:
import pandas as pd
import numpy as np

BASE = r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋"

raw = pd.read_csv(
    BASE + r"\labeled_data.csv"
)

rg3 = pd.read_csv(
    BASE + r"\moldset_labeled_rg3_다시되돌림.csv"
)

print("labeled_data :", raw.shape)
print("RG3          :", rg3.shape)

print("\n===== RG3에 없는 원본 컬럼 =====")

missing_cols = [
    c for c in raw.columns
    if c not in rg3.columns
]

print(missing_cols)

labeled_data : (7996, 45)
RG3          : (1182, 29)

===== RG3에 없는 원본 컬럼 =====
['_id', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL', 'EQUIP_CD', 'EQUIP_NAME', 'Reason', 'Switch_Over_Position', 'Barrel_Temperature_7', 'Mold_Temperature_1', 'Mold_Temperature_2', 'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'Mold_Temperature_10', 'Mold_Temperature_11', 'Mold_Temperature_12']


### 2) 추가 후보만 상태 확인

In [3]:
candidate_cols = [
    "Switch_Over_Position",
    "Barrel_Temperature_7",

    "Mold_Temperature_1",
    "Mold_Temperature_2",
    "Mold_Temperature_5",
    "Mold_Temperature_6",
    "Mold_Temperature_7",
    "Mold_Temperature_8",
    "Mold_Temperature_9",
    "Mold_Temperature_10",
    "Mold_Temperature_11",
    "Mold_Temperature_12",

    "PART_FACT_PLAN_DATE",
    "PART_FACT_SERIAL",
    "PART_NO",
    "EQUIP_CD",
    "EQUIP_NAME",

    "Shot_Number",
    "WorkingNum"
]

candidate_cols = [
    c for c in candidate_cols
    if c in raw.columns
]

candidate_cols

['Switch_Over_Position',
 'Barrel_Temperature_7',
 'Mold_Temperature_1',
 'Mold_Temperature_2',
 'Mold_Temperature_5',
 'Mold_Temperature_6',
 'Mold_Temperature_7',
 'Mold_Temperature_8',
 'Mold_Temperature_9',
 'Mold_Temperature_10',
 'Mold_Temperature_11',
 'Mold_Temperature_12',
 'PART_FACT_PLAN_DATE',
 'PART_FACT_SERIAL',
 'EQUIP_CD',
 'EQUIP_NAME']

### 3) ORIGINAL_INDEX로 원본 RG3 행 복원

In [7]:
import pandas as pd
import numpy as np

# 혹시 컬럼명에 공백이 붙어 있으면 제거
raw.columns = raw.columns.str.strip()
rg3.columns = rg3.columns.str.strip()

print("원본 컬럼:")
print(raw.columns.tolist())

print("\nRG3 컬럼:")
print(rg3.columns.tolist())

원본 컬럼:
['_id', 'TimeStamp', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL', 'PART_NAME', 'EQUIP_CD', 'EQUIP_NAME', 'PassOrFail', 'Reason', 'Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Switch_Over_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Barrel_Temperature_7', 'Hopper_Temperature', 'Mold_Temperature_1', 'Mold_Temperature_2', 'Mold_Temperature_3', 'Mold_Temperature_4', 'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'Mold_Temperature_10', 'Mold_Temperature_11', 'Mold_Temperature_12']

RG3 컬럼:
['original_index', 'PassOrFail', 'Injection_Time', 'Fillin

In [8]:
# original_index를 정수형으로
idx = rg3["original_index"].astype(int)

# 원본 데이터의 해당 행 복원
raw_rg3 = raw.iloc[idx].copy().reset_index(drop=True)

# 매칭 확인용 정보는 현재 RG3에서 붙임
raw_rg3["original_index"] = idx.values

if "RH_LH" in rg3.columns:
    raw_rg3["RH_LH"] = rg3["RH_LH"].values

print("복원 shape:", raw_rg3.shape)

복원 shape: (1182, 47)


In [9]:
if "TimeStamp" in raw_rg3.columns and "TimeStamp" in rg3.columns:

    raw_time = pd.to_datetime(
        raw_rg3["TimeStamp"]
    ).reset_index(drop=True)

    rg3_time = pd.to_datetime(
        rg3["TimeStamp"]
    ).reset_index(drop=True)

    match_rate = (
        raw_time == rg3_time
    ).mean()

    print(
        "TimeStamp 일치율:",
        match_rate
    )

TimeStamp 일치율: 1.0


### 4) 주어진 RG3에 없는 원본 컬럼

In [10]:
missing_cols = [
    col for col in raw.columns
    if col not in rg3.columns
]

print(
    "RG3에 없는 원본 컬럼:"
)

for col in missing_cols:
    print(col)

RG3에 없는 원본 컬럼:
_id
PART_FACT_PLAN_DATE
PART_FACT_SERIAL
PART_NAME
EQUIP_CD
EQUIP_NAME
Reason
Switch_Over_Position
Clamp_Open_Position
Barrel_Temperature_7
Mold_Temperature_1
Mold_Temperature_2
Mold_Temperature_5
Mold_Temperature_6
Mold_Temperature_7
Mold_Temperature_8
Mold_Temperature_9
Mold_Temperature_10
Mold_Temperature_11
Mold_Temperature_12


### 4) 추가 후보 확인

In [11]:
wanted_candidates = [
    "Switch_Over_Position",
    "Barrel_Temperature_7",

    "Mold_Temperature_1",
    "Mold_Temperature_2",
    "Mold_Temperature_5",
    "Mold_Temperature_6",
    "Mold_Temperature_7",
    "Mold_Temperature_8",
    "Mold_Temperature_9",
    "Mold_Temperature_10",
    "Mold_Temperature_11",
    "Mold_Temperature_12",

    "PART_FACT_PLAN_DATE",
    "PART_FACT_SERIAL",
    "PART_NO",
    "EQUIP_CD",
    "EQUIP_NAME",

    "Shot_Number",
    "WorkingNum"
]

candidate_cols = [
    col for col in wanted_candidates
    if col in raw_rg3.columns
]

print(
    "실제로 존재하는 후보:",
    candidate_cols
)

실제로 존재하는 후보: ['Switch_Over_Position', 'Barrel_Temperature_7', 'Mold_Temperature_1', 'Mold_Temperature_2', 'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'Mold_Temperature_10', 'Mold_Temperature_11', 'Mold_Temperature_12', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL', 'EQUIP_CD', 'EQUIP_NAME']


### 5) 후보컬럼 정보량 확인

In [12]:
candidate_summary = []

for col in candidate_cols:

    s = raw_rg3[col]

    candidate_summary.append({
        "Feature": col,
        "dtype": str(s.dtype),
        "N_Unique": s.nunique(dropna=False),
        "Missing": int(s.isna().sum()),
        "Missing_Rate": s.isna().mean(),

        "Zero_Rate": (
            (s == 0).mean()
            if pd.api.types.is_numeric_dtype(s)
            else np.nan
        )
    })

candidate_summary = (
    pd.DataFrame(candidate_summary)
    .sort_values(
        ["N_Unique", "Missing_Rate"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

candidate_summary

,Feature,dtype,N_Unique,Missing,Missing_Rate,Zero_Rate
0,PART_FACT_SERIAL,int64,6,0,0.0,0.000000
1,PART_FACT_PLAN_DATE,str,3,0,0.0,NaN
2,Switch_Over_Position,float64,2,0,0.0,0.996616
3,Barrel_Temperature_7,float64,1,0,0.0,1.000000
4,Mold_Temperature_1,float64,1,0,0.0,1.000000
5,Mold_Temperature_2,float64,1,0,0.0,1.000000
6,Mold_Temperature_5,float64,1,0,0.0,1.000000
7,Mold_Temperature_6,float64,1,0,0.0,1.000000
8,Mold_Temperature_7,float64,1,0,0.0,1.000000
9,Mold_Temperature_8,float64,1,0,0.0,1.000000


In [13]:
raw_rg3_rh = (
    raw_rg3[
        raw_rg3["RH_LH"] == "RH"
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "RH shape:",
    raw_rg3_rh.shape
)

RH shape: (591, 47)
